# Demo - Task Boundaries From the Real Import Graph
**Day 1 - Session 1, Topic 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-boundary-from-imports.ipynb)

**Goal:** Derive file ownership, shared-hub conflicts, and safe execution waves by parsing the real notification service instead of a hand-written task list.

The plan is not written into this notebook. Python's `ast` module parses every module in the course codebase, and the hub, the conflicts, and the waves are computed from that measured graph. Change the code and the plan changes with it.

> Runs offline in a temporary copy. No API key needed.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [1]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {{REPO_URL}} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

Course root: /Users/kangs/code/github/agent-orchestration-companion
Git: git version 2.50.1 (Apple Git-155)


## 2. Parse the real import graph

Read every module from disk and record which internal modules each one imports.


In [2]:
import ast
import os
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

try:
    import demo_support  # noqa: F401
except ModuleNotFoundError:
    # Setup cell above wasn't run (or the kernel restarted) - repeat its logic here.
    REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
    MARKER = Path("lab-workspace-solution") / "router.py"

    def _find_repo_root():
        directory = Path.cwd()
        for _ in range(6):
            if (directory / MARKER).exists():
                return directory
            directory = directory.parent
        clone = Path.cwd() / "agent-orchestration-companion"
        if not (clone / MARKER).exists():
            print(f"Cloning {REPO_URL} ...")
            subprocess.run(
                ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
                check=True,
                env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
            )
        return clone

    ROOT = _find_repo_root()
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT / "day1" / "demos"))


from demo_support import heading, python_files, sandbox, show_evidence, assert_true


def module_name(path, root):
    relative = path.relative_to(root).with_suffix("")
    parts = [part for part in relative.parts if part != "__init__"]
    return ".".join(parts)


def internal_imports(path, known_modules):
    tree = ast.parse(path.read_text())
    found = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module:
            candidate = node.module
        elif isinstance(node, ast.Import):
            candidate = node.names[0].name
        else:
            continue
        root_package = candidate.split(".")[0]
        for module in known_modules:
            if module == candidate or module.split(".")[0] == root_package:
                found.add(module)
    return found


# Keep one sandbox open for the whole notebook so each cell builds on the last.
sandbox_context = sandbox(with_git=False)
ROOT_COPY, _ = sandbox_context.__enter__()

files = python_files(ROOT_COPY)
MODULES = {module_name(path, ROOT_COPY): path for path in files}
GRAPH = {name: internal_imports(path, set(MODULES)) for name, path in MODULES.items()}

heading("Measured import graph (parsed, not declared)")
for name in sorted(GRAPH):
    show_evidence(f"{name}.py imports", ", ".join(sorted(GRAPH[name])) or "(none)")


Measured import graph (parsed, not declared)
--------------------------------------------
  channels.email.py imports  (none)
  channels.sms.py imports    (none)
  router.py imports          channels.email, channels.sms
  verify_record.py imports   (none)


## 3. Find the shared hub by measured fan-out

The hub is whichever module imports the most internal modules. Nothing is hardcoded.


In [3]:
HUB_NAME, hub_imports = max(GRAPH.items(), key=lambda item: len(item[1]))

heading("Shared hub detected from the graph")
show_evidence("hub module", f"{HUB_NAME}.py")
show_evidence("internal modules it wires", len(hub_imports))
show_evidence("why it is the hub", "highest internal fan-out of any module")


Shared hub detected from the graph
----------------------------------
  hub module                 router.py
  internal modules it wires  2
  why it is the hub          highest internal fan-out of any module


## 4. Compare a naive split with a contract-first plan

A naive split lets every channel task edit the hub. Watch the conflict appear.


In [7]:
CHANNEL_DIR = "channels"

channel_modules = sorted(name for name in GRAPH if name.startswith(f"{CHANNEL_DIR}."))
TASKS = [
    {"id": name.split(".")[-1], "owns": [f"{name.replace('.', '/')}.py"], "depends_on": ["schema"]}
    for name in channel_modules
]
TASKS.insert(0, {"id": "schema", "owns": ["schemas/shipment-event.json"], "depends_on": []})
TASKS.append({
    "id": "integration",
    "owns": [f"{HUB_NAME}.py"],
    "depends_on": [task["id"] for task in TASKS if task["id"] != "schema"],
})

owners = defaultdict(list)
for task in TASKS:
    if task["id"] in {"schema", "integration"}:
        continue
    for path in task["owns"] + [f"{HUB_NAME}.py"]:
        owners[path].append(task["id"])
CONFLICTS = {path: ids for path, ids in owners.items() if len(ids) > 1}

heading("Naive split: every channel task also edits the hub")
for path, ids in sorted(CONFLICTS.items()):
    show_evidence(path, f"contested by {', '.join(ids)}")
show_evidence("parallel-safe", "no: concurrent edits to one file")

heading("Contract-first plan: the hub has a single owner")
for task in TASKS:
    show_evidence(task["id"], f"owns {', '.join(task['owns'])} | after {', '.join(task['depends_on']) or '(none)'}")


Naive split: every channel task also edits the hub
--------------------------------------------------
  router.py                  contested by email, sms
  parallel-safe              no: concurrent edits to one file

Contract-first plan: the hub has a single owner
-----------------------------------------------
  schema                     owns schemas/shipment-event.json | after (none)
  email                      owns channels/email.py | after schema
  sms                        owns channels/sms.py | after schema
  integration                owns router.py | after email, sms


## 5. Compute execution waves and verify the evidence

Waves come from the derived dependencies. The checks fail loudly if a claim stops holding.


In [8]:
def execution_waves(tasks):
    remaining = {task["id"]: set(task["depends_on"]) for task in tasks}
    waves, done = [], set()
    while remaining:
        ready = sorted(task for task, deps in remaining.items() if deps <= done)
        if not ready:
            raise ValueError("Dependency cycle detected")
        waves.append(ready)
        done.update(ready)
        for task in ready:
            del remaining[task]
    return waves


WAVES = execution_waves(TASKS)
heading("Execution waves computed from the derived dependencies")
for number, wave in enumerate(WAVES, start=1):
    show_evidence(f"wave {number}", ", ".join(wave))

heading("Evidence checks")
assert_true(len(CONFLICTS) >= 1, f"the naive split contests {HUB_NAME}.py")
assert_true(sum(len(w) for w in WAVES) == len(TASKS), "every derived task is scheduled exactly once")
assert_true(len(WAVES[1]) > 1, f"wave 2 runs {len(WAVES[1])} channel tasks concurrently")
assert_true(MODULES[HUB_NAME].exists(), f"{HUB_NAME}.py was read from disk, not assumed")

sandbox_context.__exit__(None, None, None)  # remove the temporary copy
print("\nTakeaway: Parse the codebase to find the shared hub, give it one owner,")
print("and parallelism follows from the measured dependency graph.")


Execution waves computed from the derived dependencies
------------------------------------------------------
  wave 1                     schema
  wave 2                     email, sms
  wave 3                     integration

Evidence checks
---------------
  [verified] the naive split contests router.py
  [verified] every derived task is scheduled exactly once
  [verified] wave 2 runs 2 channel tasks concurrently


AssertionError: Demo evidence check failed: router.py was read from disk, not assumed

### Expected output

- The import graph shows `router.py` importing `channels.email` and `channels.sms`;
  the channel modules import nothing internal.
- `router.py` is detected as the hub, with the highest internal fan-out.
- The naive split reports `router.py` contested by `email, sms`.
- Three waves: `schema`, then `email, sms` together, then `integration`.
- Four `[verified]` lines, then the takeaway.
